# Moving-Average-Only Alarm Ablation

Raise an alarm when the six-sample moving average of the anomaly score exceeds the threshold. The consecutive-alarm requirement is not used.

The moving-average window and threshold are kept at the baseline values (`6` and `1.7085420289572733`) so this comparison removes only persistence.

In [1]:
from pathlib import Path
import sys


def _find_carl_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = _find_carl_project_root()
sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

data_dir = str(PROJECT_ROOT / "data" / "processed" / "TEP") + "/"
references_dir = str(PROJECT_ROOT / "references" / "TEP") + "/"
models_dir = str(PROJECT_ROOT / "models" / "TEP") + "/"
max_lag_selection_dir = models_dir + "max_lag_selection/"
weights_dir = max_lag_selection_dir + "weights/"
test_file_name = "TEP_Testing.parquet"
adjacency_matrix_file_name = "TEGroundTruth.txt"
max_lag = 1
window = 6
threshold = 1.7085420289572733

In [3]:
df = pd.read_parquet(data_dir + test_file_name)

n_faulty_samples = df[df["faultNumber"] != 0].shape[0]
df_faulty = df.iloc[-n_faulty_samples:]

n_samples_per_faulty_ts = df[df["faultNumber"] != 0].groupby(
    ["faultNumber", "simulationRun"]
).size().max()
n_faulty_ts = df[df["faultNumber"] != 0].groupby(["sample"]).size().max()

df_faulty = df_faulty.drop(["faultNumber", "simulationRun", "sample"], axis=1)

adjacency_matrix = np.loadtxt(references_dir + adjacency_matrix_file_name)
feature_names = df_faulty.columns
n_features = len(feature_names)
child_parents_pairs = {}

for child_idx in range(n_features):
    parents = []
    for feature_idx in range(n_features):
        if adjacency_matrix[feature_idx, child_idx] == 1:
            parents.append(feature_names[feature_idx])
    child_parents_pairs[feature_names[child_idx]] = parents

process_children = [
    child
    for child in child_parents_pairs.keys()
    if len(child_parents_pairs[child]) != 0 and int(child[1:]) < 23
]
n_process_children = len(process_children)
process_child_parents_pairs = {
    child: child_parents_pairs[child] for child in process_children
}
sample_precision = np.load(max_lag_selection_dir + "sample_precision_train.npy")

In [4]:
n_samples_test = n_faulty_ts * (n_samples_per_faulty_ts - max_lag)
res_test = np.empty((n_samples_test, n_process_children))

for child, parents in process_child_parents_pairs.items():

    df_X_test = df_faulty[parents + [child]]
    y_test = df_faulty[child]

    df_X_test_lagged = pd.concat(
        [
            df_X_test.shift(i).add_suffix(f"_lag{i}")
            for i in range(1, max_lag + 1)
        ],
        axis=1,
    )

    mask = []
    for ts_start_idx in range(0, n_faulty_samples, n_samples_per_faulty_ts):
        mask.extend(
            range(ts_start_idx + max_lag, ts_start_idx + n_samples_per_faulty_ts)
        )

    df_X_test_lagged = df_X_test_lagged.iloc[mask]
    y_test = y_test.iloc[mask].to_numpy()

    X_test = np.ones((df_X_test_lagged.shape[0], df_X_test_lagged.shape[1] + 1))
    X_test[:, 1:] = df_X_test_lagged.to_numpy()

    beta = np.load(weights_dir + f"beta_child_{child}_lag_{max_lag}.npy")
    y_test_pred = np.einsum("ij,j->i", X_test, beta, optimize=False)

    res_test[:, process_children.index(child)] = y_test - y_test_pred

scores_raw = np.einsum(
    "ij,jk,ik->i",
    res_test,
    sample_precision,
    res_test,
    optimize=False,
) / n_process_children
scores_2d = scores_raw.reshape(
    n_faulty_ts, n_samples_per_faulty_ts - max_lag
)
df_scores_2d = pd.DataFrame(scores_2d)

In [5]:
scores_2d_smoothed = df_scores_2d.T.rolling(window=window).mean().values.T
scores_2d_smoothed = scores_2d_smoothed[:, window-1:]

n_faultfree_samples_per_ts = 160
idx_first_anomaly = n_faultfree_samples_per_ts - max_lag - window + 1
y_anomaly_2d = np.ones_like(scores_2d_smoothed, dtype=int)
y_anomaly_2d[:, :idx_first_anomaly] = 0

y_anomaly_pred_2d = (scores_2d_smoothed > threshold).astype(int)

y_anomaly = y_anomaly_2d.reshape(-1)
y_anomaly_pred = y_anomaly_pred_2d.reshape(-1)

tp = np.sum((y_anomaly == 1) & (y_anomaly_pred == 1))
fp = np.sum((y_anomaly == 0) & (y_anomaly_pred == 1))
fn = np.sum((y_anomaly == 1) & (y_anomaly_pred == 0))
tn = np.sum((y_anomaly == 0) & (y_anomaly_pred == 0))

accuracy = (tp + tn) / (tp + fp + fn + tn)
tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

y_ts_anomaly_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_2d[:, 0] = y_anomaly_2d[:, :idx_first_anomaly].max(axis=1)
y_ts_anomaly_2d[:, 1] = y_anomaly_2d[:, idx_first_anomaly:].max(axis=1)
y_ts_anomaly = y_ts_anomaly_2d.reshape(-1)

y_ts_anomaly_pred_2d = np.empty((n_faulty_ts, 2))
y_ts_anomaly_pred_2d[:, 0] = y_anomaly_pred_2d[:, :idx_first_anomaly].max(axis=1)
faulty_phase_preds = y_anomaly_pred_2d[:, idx_first_anomaly:]
y_ts_anomaly_pred_2d[:, 1] = faulty_phase_preds.max(axis=1)
y_ts_anomaly_pred = y_ts_anomaly_pred_2d.reshape(-1)

tp_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 1))
fp_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 1))
fn_ts = np.sum((y_ts_anomaly == 1) & (y_ts_anomaly_pred == 0))
tn_ts = np.sum((y_ts_anomaly == 0) & (y_ts_anomaly_pred == 0))

accuracy_ts = (tp_ts + tn_ts) / (tp_ts + fp_ts + fn_ts + tn_ts)
tpr_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
fpr_ts = fp_ts / (fp_ts + tn_ts) if (fp_ts + tn_ts) > 0 else 0
precision_ts = tp_ts / (tp_ts + fp_ts) if (tp_ts + fp_ts) > 0 else 0
recall_ts = tp_ts / (tp_ts + fn_ts) if (tp_ts + fn_ts) > 0 else 0
f1_ts = 2 * precision_ts * recall_ts / (precision_ts + recall_ts) if precision_ts + recall_ts > 0 else 0

detected_mask = faulty_phase_preds.any(axis=1)
if np.any(detected_mask):
    idx_first_anomaly_pred = faulty_phase_preds[detected_mask].argmax(axis=1)
    median_idx_first_anomaly_pred = np.median(idx_first_anomaly_pred)
else:
    median_idx_first_anomaly_pred = np.nan

df_result = pd.DataFrame({
    "Alarm Logic": ["Moving average only"],
    "Moving Average Window": [window],
    "Threshold": [threshold],
    "Sample Accuracy": [accuracy],
    "Sample TPR": [tpr],
    "Sample FPR": [fpr],
    "Sample F1": [f1],
    "Trajectory-Phase Accuracy": [accuracy_ts],
    "Trajectory-Phase TPR": [tpr_ts],
    "Trajectory-Phase FPR": [fpr_ts],
    "Trajectory-Phase F1": [f1_ts],
    "Median Anomaly Detection Delay": [median_idx_first_anomaly_pred],
})

display(df_result)

,Alarm Logic,Moving Average Window,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Median Anomaly Detection Delay
0,Moving average only,6,1.708542,0.623114,0.550871,0.001597,0.710261,0.9288,0.9356,0.078,0.929281,13.0


In [6]:
from carl.tep_figure_exports import save_tep_figure_metrics
save_tep_figure_metrics(df_result, PROJECT_ROOT, 'Moving average only', 'faulty_test', 'alarm_ablations')


Saved TEP figure metrics to <repo>\reports\results\TEP\figure_metrics\alarm_ablations_moving_average_only_faulty_test.csv
